In [ ]:
%pip install -q torch torchvision pillow opencv-python segmentation-models-pytorch albumentations

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.43.1 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.115.0 which is incompatible.
gradio 5.43.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
gradio 5.43.1 requires starlette<1.0,>=0.40.0; sys_platform != "emscripten", but you have starlette 0.38.6 which is incompatible.


In [2]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

OSError: [WinError 126] The specified module could not be found. Error loading "c:\Users\abhay\anaconda3\Lib\site-packages\torch\lib\fbgemm.dll" or one of its dependencies.

In [ ]:
# ── Load trained model ──────────────────────────────────────────────────────
MODEL_PATH = "best_model.pth"   # path to saved weights

model = smp.Unet(
    encoder_name="efficientnet-b3",
    encoder_weights=None,          # weights loaded from checkpoint
    in_channels=3,
    classes=1,
    activation=None
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print("Model loaded successfully.")

In [ ]:
# ── Preprocess & run inference ───────────────────────────────────────────────
IMAGE_PATH = r"test images\test 1.png"   # test image
PATCH_SIZE  = 256

transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Load image
img_bgr = cv2.imread(IMAGE_PATH)
assert img_bgr is not None, f"Could not read image: {IMAGE_PATH}"
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

h, w = img_rgb.shape[:2]

# Pad so dimensions are divisible by PATCH_SIZE
pad_h = (PATCH_SIZE - h % PATCH_SIZE) % PATCH_SIZE
pad_w = (PATCH_SIZE - w % PATCH_SIZE) % PATCH_SIZE
img_padded = np.pad(img_rgb, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
H, W = img_padded.shape[:2]

# Stitch patch predictions into a full mask
full_mask = np.zeros((H, W), dtype=np.float32)

with torch.no_grad():
    for i in range(0, H, PATCH_SIZE):
        for j in range(0, W, PATCH_SIZE):
            patch = img_padded[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            tensor = transform(image=patch)["image"].unsqueeze(0).to(device)
            pred   = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
            full_mask[i:i+PATCH_SIZE, j:j+PATCH_SIZE] = pred

# Crop back to original size
pred_mask = (full_mask[:h, :w] > 0.5).astype(np.uint8)
print(f"Inference done. Image size: {h}×{w} | Buildings detected: {pred_mask.sum()>0}")

In [ ]:
# ── Zoning mask & illegal building detection ─────────────────────────────────
# Default zoning: right half is restricted. Modify as needed.
def create_zoning_mask(shape):
    """Returns a binary mask (1 = restricted zone)."""
    zm = np.zeros(shape, dtype=np.uint8)
    zm[:, shape[1] // 2:] = 1
    return zm

def detect_illegal_buildings(building_mask, zoning_mask):
    num_labels, labels = cv2.connectedComponents(building_mask)
    illegal, legal = [], []
    for lbl in range(1, num_labels):
        pixels = (labels == lbl)
        if (pixels & (zoning_mask == 1)).any():
            illegal.append(lbl)
        else:
            legal.append(lbl)
    return illegal, legal, labels

def overlay_illegal(image_rgb, labels, illegal_buildings):
    out = image_rgb.copy()
    for lbl in illegal_buildings:
        out[labels == lbl] = [255, 0, 0]   # red highlight
    return out

zoning_mask = create_zoning_mask(pred_mask.shape)
illegal, legal, labels = detect_illegal_buildings(pred_mask, zoning_mask)
overlay = overlay_illegal(img_rgb, labels, illegal)

print(f"Total buildings : {len(illegal) + len(legal)}")
print(f"Illegal buildings: {len(illegal)}")
print(f"Legal  buildings : {len(legal)}")

In [ ]:
# ── Visualize results ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Input Image")
axes[0].axis("off")

axes[1].imshow(pred_mask, cmap="gray")
axes[1].set_title("Building Mask")
axes[1].axis("off")

axes[2].imshow(zoning_mask, cmap="gray")
axes[2].set_title("Zoning Mask\n(white = restricted)")
axes[2].axis("off")

axes[3].imshow(overlay)
axes[3].set_title(f"Illegal Buildings (red)\nIllegal: {len(illegal)} | Legal: {len(legal)}")
axes[3].axis("off")

plt.tight_layout()
plt.show()